In [0]:
%pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [0]:
SERVICE_ACCOUNT_FILE = "/Volumes/resume_batch3/staging/metadata/batch3-dev.json"

In [0]:
from google.oauth2 import service_account
from googleapiclient.discovery import build

SCOPES = ['https://www.googleapis.com/auth/drive']

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES)

service = build('drive', 'v3', credentials=credentials)

In [0]:
FOLDER_ID = "1u6NRe-ANzl4SrZllnbtfhl3WE5hhn6s9"

query = f"'{FOLDER_ID}' in parents and trashed=false"

results = service.files().list(
    q=query,
    fields="files(id, name, mimeType)"
).execute()

files = results.get('files', [])
print(files)

In [0]:
def list_files_recursive(folder_id):
    query = f"'{folder_id}' in parents and trashed=false"

    results = service.files().list(
        q=query,
        fields="files(id, name, mimeType)"
    ).execute()

    items = results.get('files', [])
    all_files = []

    for item in items:
        if item['mimeType'] == 'application/vnd.google-apps.folder':
            all_files.extend(list_files_recursive(item['id']))
        else:
            all_files.append(item)

    return all_files

In [0]:
list_files_recursive('1u6NRe-ANzl4SrZllnbtfhl3WE5hhn6s9')

In [0]:
import io
from googleapiclient.http import MediaIoBaseDownload

# Your UC Volume path
VOLUME_PATH = "/Volumes/resume_batch3/staging/source_files/"

dbutils.fs.mkdirs(VOLUME_PATH)


def download_to_uc_volume(file_id, file_name):
    try:
        request = service.files().get_media(fileId=file_id)

        # In-memory buffer (NO disk write)
        file_stream = io.BytesIO()

        downloader = MediaIoBaseDownload(file_stream, request)

        done = False
        while not done:
            status, done = downloader.next_chunk()

        # Move pointer to beginning
        file_stream.seek(0)
       

        # ✅ Correct way for binary files
        with open(VOLUME_PATH + file_name, "wb") as f:
            f.write(file_stream.read())

        # Write directly to UC Volume
        # dbutils.fs.put(
        #     VOLUME_PATH + file_name,
        #     file_stream.read(),
        #     overwrite=True
        # )

        print(f"✅ Uploaded: {file_name}")

    except Exception as e:
        print(f"❌ Failed: {file_name} | Error: {str(e)}")

In [0]:
def process_drive_folder(folder_id):
    all_items = list_files_recursive(folder_id)

    for item in all_items:
        file_name = item['name']
        file_id = item['id']

        # Filter only PDF & DOCX
        if not (file_name.lower().endswith(".pdf") or file_name.lower().endswith(".docx")):
            continue

        download_to_uc_volume(file_id, file_name)

In [0]:
FOLDER_ID = "1u6NRe-ANzl4SrZllnbtfhl3WE5hhn6s9"

process_drive_folder(FOLDER_ID)

In [0]:
def list_files_with_path(folder_id, current_path=""):
    query = f"'{folder_id}' in parents and trashed=false"

    results = service.files().list(
        q=query,
        fields="files(id, name, mimeType)"
    ).execute()

    items = results.get('files', [])
    all_files = []

    for item in items:
        item_name = item['name']
        item_id = item['id']
        mime_type = item['mimeType']

        if mime_type == 'application/vnd.google-apps.folder':
            # 📂 Go inside folder
            new_path = f"{current_path}/{item_name}" if current_path else item_name

            all_files.extend(
                list_files_with_path(item_id, new_path)
            )

        else:
            # 📄 File with full path
            all_files.append({
                "id": item_id,
                "name": item_name,
                "path": current_path
            })

    return all_files

In [0]:
def list_files_with_path(folder_id, current_path=""):
    query = f"'{folder_id}' in parents and trashed=false"

    results = service.files().list(
        q=query,
        fields="files(id, name, mimeType)"
    ).execute()

    items = results.get('files', [])
    all_files = []

    for item in items:
        item_name = item['name']
        item_id = item['id']
        mime_type = item['mimeType']

        if mime_type == 'application/vnd.google-apps.folder':
            # 📂 Go inside folder
            new_path = f"{current_path}/{item_name}" if current_path else item_name

            all_files.extend(
                list_files_with_path(item_id, new_path)
            )

        else:
            # 📄 File with full path
            all_files.append({
                "id": item_id,
                "name": item_name,
                "path": current_path
            })

    return all_files

In [0]:
from googleapiclient.http import MediaIoBaseDownload
import os

VOLUME_BASE_PATH = "/Volumes/resume_batch3/staging/source_files/"
VOLUME_BASE_URI  = "/Volumes/resume_batch3/staging/source_files/"


def download_with_structure(file_id, file_name, folder_path):
    try:
        # 📂 Create folder path in UC Volume
        full_folder_path = os.path.join(VOLUME_BASE_PATH, folder_path)

        if folder_path:
            dbutils.fs.mkdirs(
                VOLUME_BASE_URI + folder_path
            )

        full_file_path = os.path.join(full_folder_path, file_name)

        request = service.files().get_media(fileId=file_id)

        # ✅ Stream directly to correct folder
        with open(full_file_path, "wb") as f:
            downloader = MediaIoBaseDownload(f, request)

            done = False
            while not done:
                status, done = downloader.next_chunk()

        print(f"✅ Uploaded: {folder_path}/{file_name}")

    except Exception as e:
        print(f"❌ Failed: {file_name} | Error: {str(e)}")

In [0]:
def process_drive_with_structure(folder_id):
    all_files = list_files_with_path(folder_id)

    for file in all_files:
        file_name = file['name']
        file_id = file['id']
        folder_path = file['path']

        # ✅ Filter only PDF & DOCX
        if not file_name.lower().endswith((".pdf", ".docx")):
            continue

        download_with_structure(file_id, file_name, folder_path)

In [0]:
VOLUME_DBFS_PATH = "/Volumes/resume_batch3/staging/source_files/"

In [0]:
process_drive_with_structure(FOLDER_ID)